# Multiple Comparisons

## Overview
Testing more than two groups without correction inflates the family-wise Type I error rate. With *k* tests each at α = 0.05, the probability of at least one false positive is 1 − 0.95^k — already 0.40 for 10 tests.

| Procedure | Controls | Use case |
|---|---|---|
| Tukey HSD | FWER | All pairwise, balanced ANOVA |
| Bonferroni | FWER | Any set; conservative |
| Holm | FWER | Sequential Bonferroni; always ≥ Bonferroni power |
| Dunnett | FWER | Each treatment vs one control |
| Benjamini-Hochberg | FDR | Many comparisons; exploratory |
| Planned contrasts | FWER (limited) | A priori hypotheses; most powerful |

**Quinn & Keough (2002) and Underwood (1997):** planned contrasts based on a priori hypotheses are more powerful and more scientifically defensible than post-hoc procedures applied after inspecting results.

---

In [1]:
library(tidyverse); library(multcomp); library(emmeans)
set.seed(7)
# Intertidal disturbance experiment: 5 treatments × 12 replicates
treatments <- c("Control","Low","Medium","High","Removal")
richness <- data.frame(
  treatment = factor(rep(treatments, each=12), levels=treatments),
  richness  = c(rnorm(12,18,3), rnorm(12,16,3), rnorm(12,13,3),
                rnorm(12,10,3), rnorm(12, 8,3))
)
m <- aov(richness ~ treatment, data=richness)
cat("One-way ANOVA:\n"); print(summary(m))
cat("\nSignificant F → post-hoc comparisons appropriate\n")

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'stringr' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'multcomp' was built under R version 4.4.3"
Loading required package: mvtnorm

Warning message:
"package 'm

One-way ANOVA:
            Df Sum Sq Mean Sq F value   Pr(>F)    
treatment    4 1006.9  251.72   27.95 1.09e-12 ***
Residuals   55  495.3    9.01                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Significant F → post-hoc comparisons appropriate


---
## Tukey HSD and Dunnett's test

In [2]:
# Tukey HSD — all pairwise
tukey <- TukeyHSD(m)
print(tukey)
# Compact letter display
library(multcompView)
print(multcompView::multcompLetters4(m, tukey))

cat("\n── Dunnett: each treatment vs Control only ──\n")
dunnett <- glht(m, linfct = mcp(treatment = "Dunnett"))
print(summary(dunnett))
cat("\nTukey makes", choose(5,2), "comparisons; Dunnett makes", 4,
    "→ Dunnett more powerful when only control comparisons matter\n")

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = richness ~ treatment, data = richness)

$treatment
                      diff        lwr        upr     p adj
Low-Control      -1.474750  -4.929917  1.9804163 0.7491790
Medium-Control   -5.918512  -9.373678 -2.4633453 0.0001071
High-Control     -8.994247 -12.449413 -5.5390803 0.0000000
Removal-Control -10.514187 -13.969354 -7.0590209 0.0000000
Medium-Low       -4.443762  -7.898928 -0.9885950 0.0054552
High-Low         -7.519497 -10.974663 -4.0643301 0.0000009
Removal-Low      -9.039437 -12.494604 -5.5842706 0.0000000
High-Medium      -3.075735  -6.530902  0.3794315 0.1030190
Removal-Medium   -4.595676  -8.050842 -1.1405091 0.0037469
Removal-High     -1.519941  -4.975107  1.9352260 0.7277979

$treatment
Control     Low  Medium    High Removal 
    "a"     "a"     "b"    "bc"     "c" 


── Dunnett: each treatment vs Control only ──

	 Simultaneous Tests for General Linear Hypotheses

Multiple C

---
## Bonferroni, Holm, and Benjamini-Hochberg

In [3]:
methods <- c("none","bonferroni","holm","BH")
results <- lapply(methods, function(m_adj)
  pairwise.t.test(richness$richness, richness$treatment, p.adjust.method=m_adj))
sig_counts <- sapply(results, function(r) sum(r$p.value < 0.05, na.rm=TRUE))
names(sig_counts) <- c("Uncorrected","Bonferroni","Holm","BH (FDR)")
cat("Significant pairs at alpha=0.05:\n")
print(sig_counts)
cat("\nHolm is always >= as powerful as Bonferroni (use Holm by default).\n")
cat("BH controls FDR (expected false positives among significant results)\n")
cat("  rather than FWER — more power, accepts some false positives.\n")

Significant pairs at alpha=0.05:
Uncorrected  Bonferroni        Holm    BH (FDR) 
          8           7           8           8 

Holm is always >= as powerful as Bonferroni (use Holm by default).
BH controls FDR (expected false positives among significant results)
  rather than FWER — more power, accepts some false positives.


---
## Planned (a priori) contrasts

In [4]:
# Contrasts specified BEFORE seeing the data, from biological hypotheses
# Levels: Control  Low  Medium  High  Removal
K <- rbind(
  "Removal vs all others"  = c( 4, -1, -1, -1, -1) / 4,
  "Control vs disturbed"   = c( 3, -1, -1, -1,  0) / 3,
  "Linear disturbance grad"= c(-2, -1,  0,  1,  0)   # trend L→H
)
planned <- glht(m, linfct = mcp(treatment = K))
print(summary(planned))

# Verify orthogonality
cat("\nOrthogonality check (dot products of contrast pairs):\n")
cat("  C1 · C2:", sum(K[1,]*K[2,]), "\n")
cat("  C1 · C3:", sum(K[1,]*K[3,]), "\n")
cat("  C2 · C3:", sum(K[2,]*K[3,]), "\n")
cat("  Orthogonal contrasts provide independent (non-overlapping) information.\n")
cat("\nFewer comparisons → less severe correction → more power than all-pairwise\n")


	 Simultaneous Tests for General Linear Hypotheses

Multiple Comparisons of Means: User-defined Contrasts


Fit: aov(formula = richness ~ treatment, data = richness)

Linear Hypotheses:
                             Estimate Std. Error t value Pr(>|t|)    
Removal vs all others == 0     6.7254     0.9685   6.944  < 1e-06 ***
Control vs disturbed == 0      5.4625     1.0003   5.461 2.39e-06 ***
Linear disturbance grad == 0  -7.5195     1.2251  -6.138  < 1e-06 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
(Adjusted p values reported -- single-step method)


Orthogonality check (dot products of contrast pairs):
  C1 · C2: 1.25 
  C1 · C3: -2 
  C2 · C3: -2 
  Orthogonal contrasts provide independent (non-overlapping) information.

Fewer comparisons → less severe correction → more power than all-pairwise


---
## Common Pitfalls

**1. Running post-hoc tests after a non-significant omnibus F**
Post-hoc procedures generally require a significant overall F-test. Applying Tukey HSD to a non-significant ANOVA result and mining for significant pairs is not statistically valid. Planned contrasts with strong a priori justification are an exception.

**2. Conflating Holm and Bonferroni**
The Holm procedure is a uniformly more powerful sequential extension of Bonferroni. There is no situation in which plain Bonferroni is preferable to Holm. They are frequently confused in methods sections — always specify which was used.

**3. Claiming planned contrasts were planned when they were not**
A priori contrasts must be specified from the biological hypothesis before data are examined. If you look at the data, identify the most interesting pairs, then test those "as if" they were planned, you are inflating Type I error under the guise of planned comparisons.

**4. Using TukeyHSD on unbalanced designs**
Base R `TukeyHSD()` assumes equal group sizes. For unbalanced designs use `emmeans` with `pairs(..., adjust = "tukey")` which applies the Tukey-Kramer adjustment.

**5. Confusing FWER and FDR**
FWER = probability of any false positive. FDR = expected proportion of false positives among significant results. Tukey/Bonferroni/Holm control FWER; Benjamini-Hochberg controls FDR. The right choice depends on whether you can tolerate any false positives (confirmatory) or some (exploratory).

**6. Not reporting the correction method**
"Pairwise comparisons were conducted" is incomplete. Report the test, the correction method, and the family of comparisons it was applied to (all pairwise, vs control, etc.).


---
*r_methods_library - Samantha McGarrigle*